# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of one content item for one client on one report date, for this assignment i will work with the rows where month='2026-03'

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature : impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, days_since_last_update

Label/proxy : trend_direction, used as a proxy for whether a content page is improving, stable, or declining.

Context : content_id - identifies the content item, client_id - identifies the client group, month - identifies the selected time window

Excluded : trend_pct - excluded since it directly qualifies the trend outcome and could leak information about target/proxy.
impressions_last_30d, clicks_last_30d and sessions_last_30d - excluded from initial feature set because, depending on the decision point, they can overlap with the outcome we are trying to access.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb huggingface_hub
import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

In [30]:
import os
os.environ["HF_TOKEN"] = HF_TOKEN

In [31]:
con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [32]:
rel = "hf://datasets/FlyRank/internship-warehouse"
test = con.execute(f"""
    SELECT *
    FROM read_parquet('{rel}/dim_clients.parquet')
    LIMIT 5
""").df()
test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [33]:
fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{fact_path}')
""").df()
columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


query 1: verify the grain

In [34]:
grain_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_row_keys
    FROM read_parquet('{fact_path}')
    WHERE month = '2026-03'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_row_keys
0,9841378,9841378


query 2 : row count and date window

In [35]:
window_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{fact_path}')
    WHERE month = '2026-03'
""").df()

window_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


query 3 : availability check

In [36]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS rows_with_both_available
    FROM read_parquet('{fact_path}')
    WHERE month = '2026-03'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_both_available
0,9841378,364347


five feature frame:
30-day impressions: Knowable at the decision moment because it is calculated from performance data observed before the content review decision.

30-day clicks: Knowable at the decision moment because it uses already observed search performance.

Average search position: Knowable at the decision moment because it is measured from already available GSC data.

30-day sessions: Knowable at the decision moment because it uses sessions that have already occurred.

30-day engaged sessions: Knowable at the decision moment because engagement data is available after those sessions occur and before the review decision.

In [37]:
feature_frame = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet('{fact_path}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    LIMIT 10000
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,0


In [38]:
feature_frame["needs_review"] = (
    feature_frame["ga4_engaged_sessions"] == 0
).astype(int)

feature_frame[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "needs_review"
]].head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review
0,5,0,5.400000,1,0,1
1,39,0,5.666667,2,0,1
2,179,0,5.156425,2,0,1
3,72,0,7.694444,1,0,1
4,3282,1,6.167885,1,0,1


In [39]:
model_data = feature_frame.dropna().copy()
model_data.shape

(10000, 9)

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions"
]

X = model_data[features]
y = model_data["needs_review"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

honest_predictions = model.predict(X_test)
honest_score = accuracy_score(y_test, honest_predictions)

print("Honest accuracy:", honest_score)

Honest accuracy: 0.8565


In [41]:
model_data["leaky_label_copy"] = model_data["needs_review"]

leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "leaky_label_copy"
]

X_leaky = model_data[leaky_features]
y = model_data["needs_review"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_model = RandomForestClassifier(random_state=42)
leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)
leaky_score = accuracy_score(y_test, leaky_predictions)

print("Leaky accuracy:", leaky_score)

Leaky accuracy: 1.0


In [42]:
model_data = model_data.drop(columns=["leaky_label_copy"])

print("Leaky column removed.")
print("Final honest accuracy retained:", honest_score)

Leaky column removed.
Final honest accuracy retained: 0.8565


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot prove that a change in content performance was caused by a specific action, refresh, or Google's algorithm. The historical coverage is also unbalanced because not every client or content item has the same amount of available data. Some rows may have only GSC data or GA4 data, so filtering to rows  where both are available reduces the usable sample and may exclude parts of the population. In addition, time windows can overlap, which creates a risk of leakage if information from an outcome period is accidentally used as a feature.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.